# GRACE Enhanced: Pipeline Cải Tiến trên WSL

Notebook này implement pipeline **GRACE Enhanced** — phiên bản cải tiến của bài báo
**GRACE: Empowering LLM-based Software Vulnerability Detection with Graph Structure and In-context Learning**.

## 4 Đóng góp chính:

| # | Đóng góp | Mô tả |
|---|---------|-------|
| C1 | **Tri-signal Hybrid Retrieval** | BGE-Code-v1 (semantic) + BM25 lexical + BM25 trigram syntactic → Weighted RRF |
| C2 | **Stratified Demonstration Retrieval** | Phân pool theo nhãn (vulnerable/non-vulnerable), ưu tiên lớp thiểu số |
| C3 | **DSPy + GEPA Prompt Optimization** | Tối ưu prompt tự động (placeholder — sẽ implement riêng) |
| C4 | **Bổ sung MCC** | Matthews Correlation Coefficient vào bộ metrics |

### So sánh công cụ:
| Thành phần | GRACE gốc | Cải tiến |
|---|---|---|
| Semantic Embedding | CodeT5 + BERT Whitening | **BGE-Code-v1** |
| Lexical Similarity | Jaccard | **BM25** |
| Syntactic Similarity | Levenshtein on AST | **BM25 on SimSBT trigrams** |
| Fusion | Linear (0.7α + 0.3β) | **Weighted RRF** |
| Vector Index | FAISS | **Qdrant (in-memory)** |
| AST (train) | Joern | **tree-sitter** |
| CPG (test) | Joern | Joern (giữ nguyên) |
| Metrics | Acc, P, R, F1 | Acc, P, R, F1, **MCC** |

> **Môi trường:** WSL với venv tại `/mnt/c/MY-FILES/LAB-THAY-THO/.venv`


## Bước 1: Cài đặt thư viện

Thư viện mới so với notebook gốc:
- `FlagEmbedding`: cho BGE-Code-v1 (semantic embedding)
- `qdrant-client`: cho Qdrant vector database (in-memory)
- `tree-sitter`, `tree-sitter-c`: sinh AST cho train set
- `rank_bm25`: cho BM25 scoring (lexical + syntactic)
- `scikit-learn`: cho metrics bổ sung (MCC)


In [2]:
%pip install torch transformers FlagEmbedding qdrant-client tree-sitter tree-sitter-c \
    rank_bm25 scikit-learn scipy pandas tqdm gdown google-cloud-aiplatform


Note: you may need to restart the kernel to use updated packages.


## Bước 2: Tải Dataset Devign

Dataset Devign (FFmpeg+Qemu) chứa ~27,318 hàm C/C++ đã được gắn nhãn Vulnerable/Non-vulnerable.


In [3]:
import os
import json
import random
import numpy as np

# Tải hoặc dùng lại function.json
if os.path.exists('function.json'):
    print('[OK] function.json đã có sẵn')
elif os.path.exists('../Grace-Code-Based/function.json'):
    import shutil
    shutil.copy('../Grace-Code-Based/function.json', 'function.json')
    print('[OK] Đã copy function.json từ Grace-Code-Based')
else:
    import gdown
    print('Đang tải dataset Devign từ Google Drive...')
    gdown.download(id='1x6hoF7G-tSYxg8AFybggypLZgMGDNHfF', output='function.json')
    print('[OK] Tải xong')

with open('function.json', 'r') as f:
    all_data = json.load(f)
print(f'Tổng số hàm: {len(all_data)}')
print(f'Vuln ratio: {sum(d["target"] for d in all_data)/len(all_data):.2%}')


[OK] function.json đã có sẵn
Tổng số hàm: 27318
Vuln ratio: 45.61%


## Bước 3: Chia Train/Val/Test = 8:1:1 (đúng paper)

Paper: *"We split the dataset into train, validation, and test sets with a ratio of 8:1:1."*


In [4]:
random.seed(42)

indices = list(range(len(all_data)))
random.shuffle(indices)

n = len(all_data)
n_train = int(n * 0.8)
n_val = int(n * 0.1)
n_test = n - n_train - n_val

train_data = [all_data[i] for i in indices[:n_train]]
val_data = [all_data[i] for i in indices[n_train:n_train + n_val]]
test_data = [all_data[i] for i in indices[n_train + n_val:]]

train_code_list = [d['func'] for d in train_data]
train_target_list = [d['target'] for d in train_data]
test_code_list = [d['func'] for d in test_data]
test_target_list = [d['target'] for d in test_data]

NUM_TRAIN = len(train_data)
NUM_TEST = len(test_data)

print(f'Train: {NUM_TRAIN} | Val: {len(val_data)} | Test: {NUM_TEST}')
print(f'Train vuln ratio: {sum(train_target_list)/NUM_TRAIN:.2%}')
print(f'Test vuln ratio: {sum(test_target_list)/NUM_TEST:.2%}')


Train: 21854 | Val: 2731 | Test: 2733
Train vuln ratio: 45.65%
Test vuln ratio: 44.53%


## Bước 4: Trích xuất CPG cho TEST bằng Joern

**Theo proposal:** *"Phần CPG cho prompt sẽ chỉ được sinh cho test set."*

Dùng Joern để tạo Code Property Graph (nodes + edges) cho mỗi hàm test.
Joern zip được lấy từ thư mục `Grace-Code-Based/Joern/`.


In [5]:
import subprocess
import shutil
import pandas as pd
from tqdm import tqdm
import concurrent.futures
import pickle

# === Chuẩn bị Joern ===
if not os.path.exists('Joern'):
    if os.path.exists('../Grace-Code-Based/Joern'):
        os.symlink(os.path.abspath('../Grace-Code-Based/Joern'), 'Joern')
        print('[OK] Đã tạo symlink Joern -> Grace-Code-Based/Joern')
    else:
        raise FileNotFoundError('Không tìm thấy thư mục Joern!')

joern_dir = 'Joern/joern'
if not os.path.exists(joern_dir):
    import zipfile
    print('Đang giải nén Joern...')
    with zipfile.ZipFile('Joern/joern.zip', 'r') as z:
        z.extractall('Joern/')
    print('[OK] Đã giải nén Joern')
else:
    print('[OK] Joern đã sẵn sàng')

JOERN_PARSE = './Joern/joern/joern/joern/joern-parse'
os.chmod(JOERN_PARSE, 0o755)


def extract_cpg_worker(args):
    """Worker trích xuất CPG (nodes + edges) cho 1 hàm."""
    func_code, idx = args
    tmp_dir = f'_joern_tmp_{idx}'
    out_dir = f'_joern_out_{idx}'

    for d in [tmp_dir, out_dir]:
        if os.path.exists(d):
            shutil.rmtree(d, ignore_errors=True)
    os.makedirs(tmp_dir, exist_ok=True)

    func_file = os.path.join(tmp_dir, f'func_{idx}.c')
    with open(func_file, 'w', encoding='utf-8') as f:
        f.write(func_code)

    try:
        subprocess.run([JOERN_PARSE, tmp_dir, out_dir],
                       capture_output=True, text=True, timeout=15)
    except Exception:
        for d in [tmp_dir, out_dir]:
            if os.path.exists(d):
                shutil.rmtree(d, ignore_errors=True)
        return idx, '', ''

    node_csv = os.path.join(out_dir, tmp_dir, f'func_{idx}.c', 'nodes.csv')
    edge_csv = os.path.join(out_dir, tmp_dir, f'func_{idx}.c', 'edges.csv')

    node_str, edge_str = '', ''

    if os.path.exists(node_csv):
        try:
            df = pd.read_csv(node_csv, sep='\t', on_bad_lines='skip', engine='python').fillna('')
            parts = []
            for _, row in df.iterrows():
                ntype = str(row.get('type', ''))
                ncode = str(row.get('code', '')).replace('\n', ' ').strip()
                parts.append(f'{ntype}({ncode})' if ncode else ntype)
            node_str = ' '.join(parts)
        except Exception:
            pass

    if os.path.exists(edge_csv):
        try:
            df = pd.read_csv(edge_csv, sep='\t', on_bad_lines='skip', engine='python').fillna('')
            parts = []
            for _, row in df.iterrows():
                s = str(row.get('start', ''))
                e = str(row.get('end', ''))
                t = str(row.get('type', ''))
                parts.append(f'{s}->{e}[{t}]')
            edge_str = ' '.join(parts)
        except Exception:
            pass

    for d in [tmp_dir, out_dir]:
        if os.path.exists(d):
            shutil.rmtree(d, ignore_errors=True)

    return idx, node_str, edge_str


def run_cpg_extraction(code_list, start_idx=0, max_workers=16):
    results = [None] * len(code_list)
    tasks = [(code, start_idx + i) for i, code in enumerate(code_list)]

    with concurrent.futures.ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(extract_cpg_worker, task): i for i, task in enumerate(tasks)}
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(tasks)):
            list_idx = futures[future]
            try:
                _, node_str, edge_str = future.result()
                results[list_idx] = (node_str, edge_str)
            except Exception:
                results[list_idx] = ('', '')

    return [r[0] for r in results], [r[1] for r in results]


# === Trích xuất CPG cho TEST (có cache) ===
cpg_cache = 'cpg_test_cache.pkl'
if os.path.exists(cpg_cache):
    print(f'Đang tải CPG từ cache: {cpg_cache}')
    with open(cpg_cache, 'rb') as f:
        cache = pickle.load(f)
        test_node_list = cache['test_node_list']
        test_edge_list = cache['test_edge_list']
else:
    print('=== Trích xuất CPG cho tập TEST (16 cores) ===')
    test_node_list, test_edge_list = run_cpg_extraction(
        test_code_list, start_idx=0, max_workers=16)
    with open(cpg_cache, 'wb') as f:
        pickle.dump({'test_node_list': test_node_list, 'test_edge_list': test_edge_list}, f)

print(f'[OK] Test: {sum(1 for n in test_node_list if n)}/{NUM_TEST} hàm có CPG')


Đang giải nén Joern...
[OK] Đã giải nén Joern
Đang tải CPG từ cache: cpg_test_cache.pkl
[OK] Test: 2733/2733 hàm có CPG


## Bước 5: Sinh AST + SimSBT cho TRAIN bằng tree-sitter

**Theo proposal:** *"Phần train set RAG sẽ sử dụng tree-sitter để tối ưu tốc độ thực thi."*

Dùng `tree-sitter-c` để parse mỗi hàm trong train set thành AST,
rồi sinh **SimSBT sequence** (Structure-Based Traversal trên type nodes).

SimSBT sequence sẽ được dùng cho BM25 syntactic signal ở bước sau.


In [6]:
import tree_sitter_c as tsc
from tree_sitter import Language, Parser

# === Khởi tạo tree-sitter parser cho C ===
C_LANGUAGE = Language(tsc.language())
parser = Parser(C_LANGUAGE)


def build_simsbt_treesitter(code_str):
    """
    SimSBT: Structure-Based Traversal trên type nodes của AST.
    Output: '(function_definition (parameter_list (parameter_declaration )parameter_declaration )parameter_list )function_definition'
    """
    try:
        tree = parser.parse(code_str.encode('utf-8'))
    except Exception:
        return ''

    def dfs(node, depth=0):
        if depth > 100:
            return ''
        ntype = node.type
        result = f'({ntype} '
        for child in node.children:
            result += dfs(child, depth + 1)
        result += f'){ntype} '
        return result

    return dfs(tree.root_node).strip()


# === Sinh SimSBT cho train set (có cache) ===
sbt_cache = 'train_sbt_cache.pkl'
if os.path.exists(sbt_cache):
    print(f'Đang tải SimSBT từ cache: {sbt_cache}')
    with open(sbt_cache, 'rb') as f:
        train_sbt_list = pickle.load(f)
else:
    print('Đang sinh SimSBT cho train set bằng tree-sitter...')
    train_sbt_list = []
    for code_str in tqdm(train_code_list, desc='SimSBT'):
        train_sbt_list.append(build_simsbt_treesitter(code_str))
    with open(sbt_cache, 'wb') as f:
        pickle.dump(train_sbt_list, f)

print(f'[OK] Train: {sum(1 for s in train_sbt_list if s)}/{NUM_TRAIN} hàm có SimSBT')
print(f'Ví dụ SimSBT (200 ký tự đầu): {train_sbt_list[0][:200]}...')


Đang tải SimSBT từ cache: train_sbt_cache.pkl
[OK] Train: 21854/21854 hàm có SimSBT
Ví dụ SimSBT (200 ký tự đầu): (translation_unit (function_definition (storage_class_specifier (static )static )storage_class_specifier (primitive_type )primitive_type (function_declarator (identifier )identifier (parameter_list ((...


## Bước 5.5: Sinh SimSBT cho TEST (để tính syntactic similarity)

Cần SimSBT của test functions để dùng cho BM25 syntactic query.


In [7]:
# Sinh SimSBT cho test set
sbt_test_cache = 'test_sbt_cache.pkl'
if os.path.exists(sbt_test_cache):
    print(f'Đang tải SimSBT test từ cache: {sbt_test_cache}')
    with open(sbt_test_cache, 'rb') as f:
        test_sbt_list = pickle.load(f)
else:
    print('Đang sinh SimSBT cho test set...')
    test_sbt_list = []
    for code_str in tqdm(test_code_list, desc='SimSBT-test'):
        test_sbt_list.append(build_simsbt_treesitter(code_str))
    with open(sbt_test_cache, 'wb') as f:
        pickle.dump(test_sbt_list, f)

print(f'[OK] Test: {sum(1 for s in test_sbt_list if s)}/{NUM_TEST} hàm có SimSBT')


Đang tải SimSBT test từ cache: test_sbt_cache.pkl
[OK] Test: 2733/2733 hàm có SimSBT


## Bước 6: Mã hoá mã nguồn bằng BGE-Code-v1

**Thay CodeT5 bằng BGE-Code-v1** (BAAI/bge-code-v1) — embedding model
chuyên dùng cho code, output đã được normalize sẵn.

Không cần BERT Whitening vì BGE đã output normalized vectors.


In [8]:
from FlagEmbedding import FlagModel
import torch

# === Tải BGE-Code-v1 ===
print('Đang tải BGE-Code-v1...')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
bge_model = FlagModel(
    'BAAI/bge-code-v1',
    query_instruction_for_retrieval='',  # Không cần instruction prefix cho code
    use_fp16=(DEVICE == 'cuda')
)
print(f'[OK] BGE-Code-v1 loaded on {DEVICE}')


def encode_with_bge(code_list, batch_size=32):
    """Encode danh sách code thành vectors bằng BGE-Code-v1."""
    all_vecs = []
    for i in tqdm(range(0, len(code_list), batch_size), desc='Encoding'):
        batch = code_list[i:i + batch_size]
        # Truncate mỗi code snippet để vừa context window
        batch = [c[:4000] for c in batch]
        vecs = bge_model.encode(batch)
        all_vecs.append(vecs)
    return np.vstack(all_vecs)


# === Encode (có cache) ===
bge_cache = 'bge_vectors_cache.npz'
if os.path.exists(bge_cache):
    print(f'Đang tải vectors từ cache: {bge_cache}')
    data = np.load(bge_cache)
    train_vecs = data['train_vecs']
    test_vecs = data['test_vecs']
else:
    print('Encoding train code...')
    train_vecs = encode_with_bge(train_code_list)
    print('Encoding test code...')
    test_vecs = encode_with_bge(test_code_list)
    print('Lưu vectors vào cache...')
    np.savez(bge_cache, train_vecs=train_vecs, test_vecs=test_vecs)

print(f'Train vectors: {train_vecs.shape}')
print(f'Test vectors: {test_vecs.shape}')


/mnt/c/MY-FILES/LAB-THAY-THO/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đang tải BGE-Code-v1...


Loading weights: 100%|██████████| 338/338 [00:05<00:00, 59.54it/s] 


[OK] BGE-Code-v1 loaded on cpu
Đang tải vectors từ cache: bge_vectors_cache.npz
Train vectors: (21854, 1536)
Test vectors: (2733, 1536)


## Bước 7: Xây dựng Qdrant Index + BM25 (Tri-signal)

**C1 — Tri-signal Hybrid Retrieval:**

1. **Tín hiệu ngữ nghĩa (Semantic):** BGE vectors → Qdrant dense vector index
2. **Tín hiệu từ vựng (Lexical):** BM25 trên code tokens
3. **Tín hiệu cú pháp (Syntactic):** BM25 trên SimSBT trigrams

> BM25 tính trọng số TF-IDF cho từng token, downweight các keyword phổ biến
> và upweight các identifier hiếm — phù hợp với đặc điểm phân bố token trong code.


In [9]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from rank_bm25 import BM25Okapi
import numpy as np
import pickle
import os

COLLECTION_NAME = 'devign_train'
QDRANT_DB_PATH = './qdrant_db'
BM25_CACHE = 'bm25_cache.pkl'

# 1. Khởi tạo Qdrant (Persistent thay vì memory để không mất dữ liệu)
qdrant = QdrantClient(path=QDRANT_DB_PATH)

try:
    collection_info = qdrant.get_collection(COLLECTION_NAME)
    if collection_info.points_count == NUM_TRAIN:
        print(f'[OK] Qdrant Collection `{COLLECTION_NAME}` đã tồn tại với {NUM_TRAIN} vectors. Bỏ qua upsert.')
        skip_qdrant = True
    else:
        skip_qdrant = False
except Exception:
    skip_qdrant = False

if not skip_qdrant:
    print('Đang build Qdrant Index (Semantic)...')
    qdrant.recreate_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=VEC_DIM, distance=Distance.COSINE)
    )
    points = []
    for i in tqdm(range(NUM_TRAIN), desc='Qdrant Upsert'):
        points.append(
            PointStruct(
                id=i,
                vector=train_vecs[i].tolist(),
                payload={'label': int(train_target_list[i])}
            )
        )
        if len(points) >= 1000 or i == NUM_TRAIN - 1:
            qdrant.upsert(collection_name=COLLECTION_NAME, points=points)
            points = []
    print('[OK] Đã lưu Qdrant vào ổ cứng.')

# 2. Khởi tạo BM25 (có cache)
if os.path.exists(BM25_CACHE):
    print(f'Đang tải BM25 từ cache: {BM25_CACHE}')
    with open(BM25_CACHE, 'rb') as f:
        bm25_data = pickle.load(f)
        bm25_lexical = bm25_data['lexical']
        bm25_syntactic = bm25_data['syntactic']
else:
    print('\nĐang build BM25 Index (Lexical)...')
    train_tokens = [code.split() for code in train_code_list]
    bm25_lexical = BM25Okapi(train_tokens)

    print('Đang build BM25 Index (Syntactic)...')
    train_trigrams = [sbt_to_trigrams(sbt) for sbt in train_sbt_list]
    bm25_syntactic = BM25Okapi(train_trigrams)
    
    print('\nĐang lưu BM25 vào ổ cứng...')
    with open(BM25_CACHE, 'wb') as f:
        pickle.dump({'lexical': bm25_lexical, 'syntactic': bm25_syntactic}, f)
    print('[OK] Đã lưu xong BM25.')

print('\n[OK] Đã build xong tất cả Indexes (Qdrant + BM25x2) sẵn sàng cho Retrieval!')


Khởi tạo Qdrant in-memory...


Qdrant upsert:  91%|█████████ | 20/22 [00:17<00:01,  1.08it/s]/tmp/ipykernel_504448/4084714704.py:32: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 21000 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  qdrant.upsert(collection_name=COLLECTION_NAME, points=points)
Qdrant upsert: 100%|██████████| 22/22 [00:18<00:00,  1.19it/s]


[OK] Qdrant: 21854 vectors indexed

Xây dựng BM25 Lexical Index...
[OK] BM25 Lexical: 21854 documents
Xây dựng BM25 Syntactic Index (SimSBT trigrams)...
[OK] BM25 Syntactic: 21854 documents


## Bước 8: Stratified Demonstration Retrieval (C1 + C2)

### C1 — Weighted Reciprocal Rank Fusion (RRF):
$$\text{RRF}(d) = \sum_{s \in \{semantic, lexical, syntactic\}} \frac{w_s}{k + r_s(d)}$$

Trong đó $w_{semantic}$ > $w_{lexical}$ = $w_{syntactic}$ (ưu tiên tín hiệu ngữ nghĩa).

### C2 — Stratified Retrieval:
Phân pool theo nhãn → truy xuất top-k từ mỗi pool → ghép theo tỷ lệ ưu tiên lớp thiểu số.

> *"Tập retrieval pool được phân chia thành hai pool độc lập theo nhãn (vulnerable / non-vulnerable).
> Với mỗi mẫu cần phân tích, hệ thống truy xuất top-k theo Weighted RRF score trong từng pool
> riêng biệt, sau đó ghép kết quả theo tỷ lệ kiểm soát."*


In [13]:
import pickle
import os
cache_file = 'stratified_retrieval_cache.pkl'

demonstrations = []
start_idx = 0
if os.path.exists(cache_file):
    print(f'Đang tải kết quả Retrieval từ cache: {cache_file}')
    with open(cache_file, 'rb') as f:
        demonstrations = pickle.load(f)
    start_idx = len(demonstrations)
    print(f'Đã có sẵn {start_idx}/{NUM_TEST} kết quả. Đang tiếp tục...')

if start_idx < NUM_TEST:
    # === Phân pool theo nhãn (C2) ===
    vuln_indices = [i for i in range(NUM_TRAIN) if train_target_list[i] == 1]
    nonvuln_indices = [i for i in range(NUM_TRAIN) if train_target_list[i] == 0]
    if start_idx == 0:
        print(f'Pool Vulnerable: {len(vuln_indices)} hàm')
        print(f'Pool Non-vulnerable: {len(nonvuln_indices)} hàm')
    
    # === Weighted RRF parameters ===
    W_SEMANTIC = 0.5
    W_LEXICAL = 0.25
    W_SYNTACTIC = 0.25
    RRF_K = 60
    TOP_K_PER_SIGNAL = 50
    
    from qdrant_client.models import Filter, FieldCondition, MatchValue
    
    def weighted_rrf(semantic_ranks, lexical_ranks, syntactic_ranks, candidate_ids):
        scores = {}
        for cid in candidate_ids:
            sem_r = semantic_ranks.get(cid, 9999)
            lex_r = lexical_ranks.get(cid, 9999)
            syn_r = syntactic_ranks.get(cid, 9999)
            scores[cid] = W_SEMANTIC / (RRF_K + sem_r) + W_LEXICAL / (RRF_K + lex_r) + W_SYNTACTIC / (RRF_K + syn_r)
        return scores
    
    def retrieve_from_pool_optimized(test_idx, pool_indices, pool_label, lex_scores, syn_scores, top_k=5):
        qdrant_results = qdrant.query_points(
            collection_name=COLLECTION_NAME,
            query=test_vecs[test_idx].tolist(),
            limit=TOP_K_PER_SIGNAL,
            query_filter=Filter(
                must=[FieldCondition(key='label', match=MatchValue(value=pool_label))]
            )
        )
        sem_candidates = [(r.id, r.score) for r in getattr(qdrant_results, 'points', qdrant_results)]
        semantic_ranks = {cid: rank for rank, (cid, _) in enumerate(sem_candidates)}
    
        lex_pool_scores = [(idx, lex_scores[idx]) for idx in pool_indices]
        lex_pool_scores.sort(key=lambda x: x[1], reverse=True)
        lexical_ranks = {cid: rank for rank, (cid, _) in enumerate(lex_pool_scores[:TOP_K_PER_SIGNAL])}
    
        syn_pool_scores = [(idx, syn_scores[idx]) for idx in pool_indices]
        syn_pool_scores.sort(key=lambda x: x[1], reverse=True)
        syntactic_ranks = {cid: rank for rank, (cid, _) in enumerate(syn_pool_scores[:TOP_K_PER_SIGNAL])}
    
        all_candidates = set(semantic_ranks.keys()) | set(lexical_ranks.keys()) | set(syntactic_ranks.keys())
        rrf_scores = weighted_rrf(semantic_ranks, lexical_ranks, syntactic_ranks, all_candidates)
    
        sorted_candidates = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
        return sorted_candidates[:top_k]
    
    print('\nĐang thực hiện Stratified Demonstration Retrieval (Optimized & Checkpointed)...')
    
    for i in tqdm(range(start_idx, NUM_TEST), desc='Retrieval', initial=start_idx, total=NUM_TEST):
        test_tokens = list(set(test_code_list[i].split()))[:300]
        test_trigrams = list(set(sbt_to_trigrams(test_sbt_list[i])))[:300]
        
        lex_scores = bm25_lexical.get_scores(test_tokens)
        syn_scores = bm25_syntactic.get_scores(test_trigrams)
        
        vuln_top = retrieve_from_pool_optimized(i, vuln_indices, pool_label=1, lex_scores=lex_scores, syn_scores=syn_scores, top_k=1)
        nonvuln_top = retrieve_from_pool_optimized(i, nonvuln_indices, pool_label=0, lex_scores=lex_scores, syn_scores=syn_scores, top_k=1)
    
        demos = []
        if vuln_top:
            idx = vuln_top[0][0]
            demos.append({'code': train_code_list[idx], 'label': 'Vulnerable', 'score': vuln_top[0][1]})
        if nonvuln_top:
            idx = nonvuln_top[0][0]
            demos.append({'code': train_code_list[idx], 'label': 'Non-vulnerable', 'score': nonvuln_top[0][1]})
    
        demonstrations.append(demos)
    
        # Checkpoint mỗi 100 bước
        if (i + 1) % 100 == 0:
            with open(cache_file, 'wb') as f:
                pickle.dump(demonstrations, f)
    
    # Lưu cache chốt hạ ở cuối cùng
    with open(cache_file, 'wb') as f:
        pickle.dump(demonstrations, f)

print(f'\n[OK] Đã retrieval demonstrations cho {len(demonstrations)} test functions')
print(f'Avg demos per test: {np.mean([len(d) for d in demonstrations]):.1f}')


Pool Vulnerable: 9977 hàm
Pool Non-vulnerable: 11877 hàm

Đang thực hiện Stratified Demonstration Retrieval (Optimized)...


Retrieval:   0%|          | 0/2733 [00:00<?, ?it/s]

Retrieval: 100%|██████████| 2733/2733 [4:08:04<00:00,  5.45s/it]  


[OK] Đã retrieval demonstrations cho 2733 test functions
Avg demos per test: 2.0


## Bước 9: Ghép dữ liệu vào JSON

Tạo `devign_test_processed.json` với format mở rộng:
- `func`: mã nguồn hàm
- `target`: nhãn thực (0 hoặc 1)
- `node`: thông tin node từ CPG (Joern)
- `edge`: thông tin edge từ CPG (Joern)
- `demonstrations`: list các examples kèm label (stratified)


In [14]:
output_data = []
for i in range(NUM_TEST):
    record = {
        'func': test_code_list[i],
        'target': test_target_list[i],
        'node': test_node_list[i],
        'edge': test_edge_list[i],
        'demonstrations': demonstrations[i],
        # Backward-compatible: cũng lưu example đơn (demo đầu tiên)
        'example': demonstrations[i][0]['code'] if demonstrations[i] else '',
        'example_label': demonstrations[i][0]['label'] if demonstrations[i] else '',
    }
    output_data.append(record)

with open('devign_test_processed.json', 'w', encoding='utf-8') as f:
    json.dump(output_data, f)

print(f'[OK] Đã tạo devign_test_processed.json với {len(output_data)} hàm')
print(f'Vuln: {sum(1 for d in output_data if d["target"] == 1)}')
print(f'Non-vuln: {sum(1 for d in output_data if d["target"] == 0)}')
print(f'Avg demonstrations per function: {np.mean([len(d["demonstrations"]) for d in output_data]):.1f}')


[OK] Đã tạo devign_test_processed.json với 2733 hàm
Vuln: 1217
Non-vuln: 1516
Avg demonstrations per function: 2.0


## Bước 10: Kiểm tra kết nối Vertex AI

Paper gốc dùng GPT-4. Ta thay bằng **Qwen3-Coder-480B** qua Vertex AI.

> **Lưu ý:** Nếu bước này báo lỗi, hãy mở Terminal WSL và chạy:
> `gcloud auth application-default login`


In [15]:
import vertexai
from vertexai.generative_models import GenerativeModel

vertexai.init(project='grace-enhanced', location='global')
llm_model = GenerativeModel('publishers/qwen/models/qwen3-coder-480b-a35b-instruct-maas')

try:
    response = llm_model.generate_content('Say hello!')
    print(f'Response: {response.text}')
    print('[OK] Vertex AI đã sẵn sàng')
except Exception as e:
    print(f'[LỖI] {e}')
    print('Hãy kiểm tra lại project ID và quyền truy cập.')


/mnt/c/MY-FILES/LAB-THAY-THO/.venv/lib/python3.12/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Response: Hello! It's nice to meet you! How are you doing today?
[OK] Vertex AI đã sẵn sàng


## Bước 11: Tạo basep.py và llmpre.py

Hai file chạy inference:
- `basep.py`: Baseline — chỉ code + prompt (không graph, không example)
- `llmpre.py`: **GRACE Enhanced** — code + graph + stratified demonstrations + labels

**Cải tiến so với notebook cũ:**
- Gửi kèm **label** của demonstration (đúng paper)
- Gửi **2 demonstrations** (1 vuln + 1 non-vuln) — Stratified (C2)
- Tính thêm **MCC** (C4)
- Dùng keyword matching để trích xuất kết quả (robust)


In [ ]:
# === basep.py (Baseline) ===
basep_code = '''import vertexai
from vertexai.generative_models import GenerativeModel
import json
import csv
import logging
import math

vertexai.init(project="grace-enhanced", location="global")
model = GenerativeModel("publishers/qwen/models/qwen3-coder-480b-a35b-instruct-maas")

templates = {
    1: ("In the above code snippet, check for potential security vulnerabilities "
        "and output either 'Vulnerable' or 'Non-vulnerable'. "
        "You are now an excellent programmer."
        "You are conducting a function vulnerability detection task for C/C++ language."),
}

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
fh = logging.FileHandler('devignmetrics_basep.log')
fh.setFormatter(logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s'))
logger.addHandler(fh)


def extract_prediction(text):
    text_lower = text.strip().lower()
    if 'non-vulnerable' in text_lower or 'non vulnerable' in text_lower:
        return 0
    elif 'vulnerable' in text_lower:
        return 1
    text_clean = text.strip().replace('`', '').strip()
    if text_clean == '0':
        return 0
    elif text_clean == '1':
        return 1
    return 2


def calculate_mcc(preds, truths):
    """Matthews Correlation Coefficient (C4)."""
    tp = sum(1 for p, t in zip(preds, truths) if p == t == 1)
    tn = sum(1 for p, t in zip(preds, truths) if p == t == 0)
    fp = sum(1 for p, t in zip(preds, truths) if p == 1 and t == 0)
    fn = sum(1 for p, t in zip(preds, truths) if p == 0 and t == 1)
    denom = math.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    if denom == 0:
        return 0.0
    return (tp*tn - fp*fn) / denom


def main():
    with open('devign_test_processed.json', 'r') as f:
        data = json.load(f)

    def calculate_metrics(preds, truths):
        tp = sum(1 for p, t in zip(preds, truths) if p == t == 1)
        tn = sum(1 for p, t in zip(preds, truths) if p == t == 0)
        fp = sum(1 for p, t in zip(preds, truths) if p == 1 and t == 0)
        fn = sum(1 for p, t in zip(preds, truths) if p == 0 and t == 1)
        n = len(preds)
        acc = (tp + tn) / n if n else 0
        prec = tp / (tp + fp) if (tp + fp) else 0
        rec = tp / (tp + fn) if (tp + fn) else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
        return acc, prec, rec, f1

    prediction_ls = []
    ground_truth = []

    for row in data:
        inputCode = row['func'][:4000]
        prompt = inputCode + templates[1]

        try:
            response = model.generate_content(prompt)
            raw = response.text.strip()
        except Exception as e:
            logger.error(f"Error: {e}")
            raw = ""

        prediction = extract_prediction(raw)
        print(f"Raw: {raw[:80]}... => {prediction}")

        prediction_ls.append(prediction)
        ground_truth.append(row['target'])

        with open('devignresults_basep.csv', 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['Prediction', 'Groundtruth'])
            writer.writerows(zip(prediction_ls, ground_truth))

        acc, prec, rec, f1 = calculate_metrics(prediction_ls, ground_truth)
        mcc = calculate_mcc(prediction_ls, ground_truth)
        msg = f"[{len(prediction_ls)}/{len(data)}] Acc:{acc:.4f} P:{prec:.4f} R:{rec:.4f} F1:{f1:.4f} MCC:{mcc:.4f}"
        print(msg)
        logger.info(msg)


if __name__ == '__main__':
    main()
'''

with open('basep.py', 'w', encoding='utf-8') as f:
    f.write(basep_code)
print('[OK] Đã tạo basep.py')

# === llmpre.py (GRACE Enhanced) ===
llmpre_code = '''import vertexai
from vertexai.generative_models import GenerativeModel
import json
import csv
import logging
import math

vertexai.init(project="grace-enhanced", location="global")
model = GenerativeModel("publishers/qwen/models/qwen3-coder-480b-a35b-instruct-maas")

templates = {
    1: ("In the above code snippet, check for potential security vulnerabilities "
        "and output either 'Vulnerable' or 'Non-vulnerable'. "
        "You are now an excellent programmer."
        "You are conducting a function vulnerability detection task for C/C++ language."),
    2: "The node information of the function is as follows:",
    3: "The edge information of the function is as follows:",
    4: "Here is an example for you to learn from:",
}

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
fh = logging.FileHandler('devignmetrics_llmpre.log')
fh.setFormatter(logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s'))
logger.addHandler(fh)


def extract_prediction(text):
    text_lower = text.strip().lower()
    if 'non-vulnerable' in text_lower or 'non vulnerable' in text_lower:
        return 0
    elif 'vulnerable' in text_lower:
        return 1
    text_clean = text.strip().replace('`', '').strip()
    if text_clean == '0':
        return 0
    elif text_clean == '1':
        return 1
    return 2


def calculate_mcc(preds, truths):
    """Matthews Correlation Coefficient (C4)."""
    tp = sum(1 for p, t in zip(preds, truths) if p == t == 1)
    tn = sum(1 for p, t in zip(preds, truths) if p == t == 0)
    fp = sum(1 for p, t in zip(preds, truths) if p == 1 and t == 0)
    fn = sum(1 for p, t in zip(preds, truths) if p == 0 and t == 1)
    denom = math.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    if denom == 0:
        return 0.0
    return (tp*tn - fp*fn) / denom


def main():
    with open('devign_test_processed.json', 'r') as f:
        data = json.load(f)

    def calculate_metrics(preds, truths):
        tp = sum(1 for p, t in zip(preds, truths) if p == t == 1)
        tn = sum(1 for p, t in zip(preds, truths) if p == t == 0)
        fp = sum(1 for p, t in zip(preds, truths) if p == 1 and t == 0)
        fn = sum(1 for p, t in zip(preds, truths) if p == 0 and t == 1)
        n = len(preds)
        acc = (tp + tn) / n if n else 0
        prec = tp / (tp + fp) if (tp + fp) else 0
        rec = tp / (tp + fn) if (tp + fn) else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
        return acc, prec, rec, f1

    prediction_ls = []
    ground_truth = []

    for row in data:
        inputCode = row['func'][:4000]
        inputnode = row.get('node', '')[:2000]
        inputedge = row.get('edge', '')[:2000]

        # === C2: Stratified Demonstrations ===
        demos = row.get('demonstrations', [])
        demo_text = ""
        for demo in demos:
            demo_code = demo.get('code', '')[:3000]
            demo_label = demo.get('label', '')
            demo_text += f"""\n{templates[4]}\n{demo_code}\nThis example is {demo_label}.\n"""

        # Fallback: backward-compatible single example
        if not demo_text:
            inputex = row.get('example', '')[:4000]
            example_label = row.get('example_label', '')
            demo_text = f"""\n{templates[4]}\n{inputex}\nThis example is {example_label}.\n"""

        # === Prompt: code + task + graph + demonstrations ===
        prompt = (
            inputCode + templates[1]
            + templates[2] + inputnode
            + templates[3] + inputedge
            + demo_text
        )

        try:
            response = model.generate_content(prompt)
            raw = response.text.strip()
        except Exception as e:
            logger.error(f"Error: {e}")
            raw = ""

        prediction = extract_prediction(raw)
        print(f"Raw: {raw[:80]}... => {prediction}")

        prediction_ls.append(prediction)
        ground_truth.append(row['target'])

        with open('devignresults_llmpre.csv', 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['Prediction', 'Groundtruth'])
            writer.writerows(zip(prediction_ls, ground_truth))

        acc, prec, rec, f1 = calculate_metrics(prediction_ls, ground_truth)
        mcc = calculate_mcc(prediction_ls, ground_truth)
        msg = f"[{len(prediction_ls)}/{len(data)}] Acc:{acc:.4f} P:{prec:.4f} R:{rec:.4f} F1:{f1:.4f} MCC:{mcc:.4f}"
        print(msg)
        logger.info(msg)


if __name__ == '__main__':
    main()
'''

with open('llmpre.py', 'w', encoding='utf-8') as f:
    f.write(llmpre_code)
print('[OK] Đã tạo llmpre.py')


[OK] Đã tạo basep.py
[OK] Đã tạo llmpre.py


## Bước 12: Chạy Baseline (`basep.py`)

Chạy mô hình cơ sở: chỉ gửi **mã nguồn + prompt cơ bản** cho LLM.

> Đây là phiên bản **không dùng graph và example** — baseline để so sánh.


In [17]:
%run basep.py


/mnt/c/MY-FILES/LAB-THAY-THO/.venv/lib/python3.12/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Raw: **Non-vulnerable**

### Explanation:

This function `loop_filter` is part of the... => 0
[1/2733] Acc:1.0000 P:0.0000 R:0.0000 F1:0.0000 MCC:0.0000
Raw: Looking at this code, I can identify several potential security vulnerabilities:... => 1
[2/2733] Acc:1.0000 P:1.0000 R:1.0000 F1:1.0000 MCC:1.0000
Raw: Looking at this code snippet, I need to analyze it for potential security vulner... => 0
[3/2733] Acc:1.0000 P:1.0000 R:1.0000 F1:1.0000 MCC:1.0000
Raw: **Non-vulnerable**

### Explanation:

This function `vfio_early_setup_msix` is p... => 0
[4/2733] Acc:0.7500 P:1.0000 R:0.5000 F1:0.6667 MCC:0.5774
Raw: Looking at this code snippet, I need to analyze it for potential security vulner... => 1
[5/2733] Acc:0.8000 P:1.0000 R:0.6667 F1:0.8000 MCC:0.6667
Raw: Looking at this JSON parser code, I need to analyze it for potential security vu... => 1
[6/2733] Acc:0.6667 P:0.6667 R:0.6667 F1:0.6667 MCC:0.3333
Raw: Looking at this code snippet, I need to analyze it for potential security vul

## Bước 13: Chạy GRACE Enhanced (`llmpre.py`)

Chạy pipeline đầy đủ với các cải tiến:
- **C1:** Tri-signal Hybrid Retrieval (BGE + BM25 + Weighted RRF)
- **C2:** Stratified Demonstrations (1 vuln + 1 non-vuln)
- **C4:** MCC metric

> So sánh kết quả với Baseline ở bước trước.


In [1]:
%run llmpre.py


/mnt/c/MY-FILES/LAB-THAY-THO/.venv/lib/python3.12/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Raw: Looking at this H.264 loop filter function, I need to analyze it for potential s... => 0
[1/2733] Acc:1.0000 P:0.0000 R:0.0000 F1:0.0000 MCC:0.0000
Raw: Looking at this code, I need to analyze it for potential security vulnerabilitie... => 1
[2/2733] Acc:1.0000 P:1.0000 R:1.0000 F1:1.0000 MCC:1.0000
Raw: Looking at the code snippet for `qio_channel_socket_listen_async`, I need to ana... => 1
[3/2733] Acc:0.6667 P:0.5000 R:1.0000 F1:0.6667 MCC:0.5000
Raw: Looking at this code, I need to analyze it for potential security vulnerabilitie... => 1
[4/2733] Acc:0.7500 P:0.6667 R:1.0000 F1:0.8000 MCC:0.5774
Raw: Looking at the code snippet for `qpci_msix_masked`, I need to analyze it for pot... => 1
[5/2733] Acc:0.8000 P:0.7500 R:1.0000 F1:0.8571 MCC:0.6124
Raw: Looking at the code, I need to analyze it for potential security vulnerabilities... => 0
[6/2733] Acc:0.8333 P:0.7500 R:1.0000 F1:0.8571 MCC:0.7071
Raw: Looking at this code, I need to analyze it for potential security vulnerabili

## Bước 14: So sánh kết quả (Acc, P, R, F1, MCC)

Đọc kết quả từ CSV và tính toán metrics cuối cùng cho cả Baseline và GRACE Enhanced.

**C4 — MCC (Matthews Correlation Coefficient):**
$$MCC = \frac{TP \times TN - FP \times FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

MCC ∈ [-1, 1] — giá trị 0 = random, 1 = perfect, -1 = inverse.


In [4]:
import math
from sklearn.metrics import matthews_corrcoef, accuracy_score, precision_score, recall_score, f1_score


def load_results(csv_path):
    """Đọc kết quả từ CSV file."""
    preds, truths = [], []
    with open(csv_path, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            p = int(row['Prediction'])
            t = int(row['Groundtruth'])
            if p in (0, 1):  # Bỏ qua prediction = 2 (invalid)
                preds.append(p)
                truths.append(t)
    return preds, truths


def compute_all_metrics(preds, truths):
    """Tính tất cả metrics bao gồm MCC (C4)."""
    return {
        'Accuracy': accuracy_score(truths, preds),
        'Precision': precision_score(truths, preds, zero_division=0),
        'Recall': recall_score(truths, preds, zero_division=0),
        'F1': f1_score(truths, preds, zero_division=0),
        'MCC': matthews_corrcoef(truths, preds),
    }


# === Load and compare ===
results = {}

for name, csv_path in [('Baseline', 'devignresults_basep.csv'),
                        ('GRACE Enhanced', 'devignresults_llmpre.csv')]:
    if os.path.exists(csv_path):
        preds, truths = load_results(csv_path)
        metrics = compute_all_metrics(preds, truths)
        results[name] = metrics
        print(f'\n=== {name} ({len(preds)} valid predictions) ===')
        for k, v in metrics.items():
            print(f'  {k}: {v:.4f}')
    else:
        print(f'[SKIP] {csv_path} chưa tồn tại')

# === Bảng so sánh ===
if len(results) == 2:
    print('\n' + '='*70)
    print(f'{"Metric":<15} {"Baseline":>12} {"GRACE Enhanced":>16} {"Δ":>10}')
    print('-'*70)
    for metric in ['Accuracy', 'Precision', 'Recall', 'F1', 'MCC']:
        base = results['Baseline'][metric]
        enhanced = results['GRACE Enhanced'][metric]
        delta = enhanced - base
        arrow = '↑' if delta > 0 else ('↓' if delta < 0 else '=')
        print(f'{metric:<15} {base:>12.4f} {enhanced:>16.4f} {delta:>+9.4f} {arrow}')
    print('='*70)



=== Baseline (2733 valid predictions) ===
  Accuracy: 0.5357
  Precision: 0.4791
  Recall: 0.4897
  F1: 0.4844
  MCC: 0.0622

=== GRACE Enhanced (2733 valid predictions) ===
  Accuracy: 0.5199
  Precision: 0.4601
  Recall: 0.4503
  F1: 0.4551
  MCC: 0.0262

Metric              Baseline   GRACE Enhanced          Δ
----------------------------------------------------------------------
Accuracy              0.5357           0.5199   -0.0157 ↓
Precision             0.4791           0.4601   -0.0190 ↓
Recall                0.4897           0.4503   -0.0394 ↓
F1                    0.4844           0.4551   -0.0292 ↓
MCC                   0.0622           0.0262   -0.0360 ↓


## [Placeholder] Bước 15: DSPy + GEPA Prompt Optimization (C3)

**TODO:** Implement DSPy pipeline với GEPA optimizer.

Ý tưởng:
1. Khai báo pipeline phát hiện lỗ hổng dưới dạng chương trình DSPy
2. Các thành phần prompt (Pb, Pi, Pd) trở thành tham số tối ưu hóa được
3. Dùng GEPA (Genetic-Pareto optimizer) với CWE-aware textual feedback
4. Tối ưu trên validation set rồi đánh giá trên test set

```python
# import dspy
# from dspy.teleprompt import GEPA
#
# class VulnDetector(dspy.Module):
#     def __init__(self):
#         self.detect = dspy.ChainOfThought('code, graph_info, demonstrations -> vulnerability_label')
#
#     def forward(self, code, graph_info, demonstrations):
#         return self.detect(code=code, graph_info=graph_info, demonstrations=demonstrations)
#
# optimizer = GEPA(metric=cwe_aware_feedback_metric, ...)
# optimized_detector = optimizer.compile(VulnDetector(), trainset=val_data)
```

> Sẽ implement trong notebook riêng khi có đủ thời gian chạy optimization trên validation set.
